In [19]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, make_scorer, recall_score
from sklearn.model_selection import (
    GridSearchCV,
    StratifiedKFold,
    cross_validate,
    train_test_split,
)
from sklearn.preprocessing import LabelEncoder

from sklearn.preprocessing import LabelEncoder
import pandas as pd

In [20]:
def find_txt_files(training_root_path: Path) -> list[Path]:
    """Find all .txt files recursively under the training folder.
    Args:
            training_root_path: Root path for the training data directory.
    Returns:
            A sorted list of .txt file paths.
    """
    return sorted(training_root_path.rglob("*.txt"))


def parse_patient_txt_file(txt_file_path: Path) -> dict[str, str]:
    """Parse a patient text file of `Key: Value` rows.
    Args:
            txt_file_path: Path to one patient .txt file.
    Returns:
            A dictionary where keys are field names and values are raw string values.
    Raises:
            OSError: If the file cannot be read.
    """
    record: dict[str, str] = {}
    for raw_line in txt_file_path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line:
            continue
        if ":" not in line:
            continue
        key, value = line.split(":", maxsplit=1)
        record[key.strip()] = value.strip()
    record["source_file"] = str(txt_file_path)
    return record


def normalize_string_missing_values(training_dataframe: pd.DataFrame) -> pd.DataFrame:
    """Normalize literal string missing markers to pandas missing values.
    Args:
            training_dataframe: Raw dataframe created from patient text files or parquet.
    Returns:
            A dataframe where common string missing markers are converted to ``pd.NA``.
    """
    normalized_dataframe = training_dataframe.copy()
    missing_markers = {
        "": pd.NA,
        "nan": pd.NA,
        "NaN": pd.NA,
        "NAN": pd.NA,
        "none": pd.NA,
        "None": pd.NA,
        "NONE": pd.NA,
        "null": pd.NA,
        "Null": pd.NA,
        "NULL": pd.NA,
    }
    return normalized_dataframe.replace(missing_markers)


def build_training_dataframe(training_root_path: Path) -> pd.DataFrame:
    """Load all patient text files into one pandas DataFrame.
    Args:
            training_root_path: Root path that contains patient folders and .txt files.
    Returns:
            A DataFrame with one row per file and one column per text field.
    """
    records: list[dict[str, str]] = []
    for txt_file_path in find_txt_files(training_root_path):
        records.append(parse_patient_txt_file(txt_file_path))
    training_dataframe = pd.DataFrame.from_records(records)
    training_dataframe = normalize_string_missing_values(training_dataframe)
    return convert_column_types(training_dataframe)


def convert_column_types(training_dataframe: pd.DataFrame) -> pd.DataFrame:
    """Convert known columns to analysis-friendly dtypes.
    Args:
            training_dataframe: DataFrame created from patient text files.
    Returns:
            A DataFrame with converted numeric and boolean columns.
    """
    converted_dataframe = normalize_string_missing_values(training_dataframe)
    for numeric_column in ["Age", "TTM", "CPC", "ROSC"]:
        if numeric_column in converted_dataframe.columns:
            converted_dataframe[numeric_column] = pd.to_numeric(
                converted_dataframe[numeric_column],
                errors="coerce",
            )
    for bool_column in ["OHCA", "Shockable Rhythm"]:
        if bool_column in converted_dataframe.columns:
            converted_dataframe[bool_column] = converted_dataframe[bool_column].map(
                {"True": True, "False": False}
            )
    return converted_dataframe


def add_missing_value_summary(training_dataframe: pd.DataFrame) -> pd.DataFrame:
    """Summarize missing values after string normalization.
    Args:
            training_dataframe: Clean dataframe used for analysis.
    Returns:
            A summary table with counts and percentages of missing values.
    """
    missing_counts = training_dataframe.isna().sum()
    missing_percentages = (missing_counts / len(training_dataframe) * 100).round(2)
    return pd.DataFrame(
        {
            "Missing count": missing_counts,
            "Missing percentage": missing_percentages,
        }
    )


def prepare_modeling_features(
    input_dataframe: pd.DataFrame,
    numeric_fill_values: dict[str, float] | None = None,
) -> tuple[pd.DataFrame, dict[str, float]]:
    """Create model-ready features that keep rows with recorded missingness.
    Args:
            input_dataframe: Input dataframe containing the clinical predictors.
            numeric_fill_values: Optional fill values learned from training data.
    Returns:
            A tuple of the prepared dataframe and the numeric fill values used.
    """
    modeling_dataframe = input_dataframe.copy()
    if "Sex" in modeling_dataframe.columns:
        modeling_dataframe["Sex"] = modeling_dataframe["Sex"].fillna("Unknown")
    for bool_column in ["OHCA", "Shockable Rhythm"]:
        if bool_column in modeling_dataframe.columns:
            modeling_dataframe[bool_column] = (
                modeling_dataframe[bool_column]
                .map({True: "Yes", False: "No"})
                .fillna("Not recorded")
            )
    if "TTM" in modeling_dataframe.columns:
        modeling_dataframe["TTM"] = (
            modeling_dataframe["TTM"].map({33.0: "33C", 36.0: "36C"}).fillna("No TTM")
        )
    if numeric_fill_values is None:
        numeric_fill_values = {}
        for numeric_column in ["Age", "ROSC"]:
            if numeric_column in modeling_dataframe.columns:
                numeric_fill_values[numeric_column] = float(
                    modeling_dataframe[numeric_column].median()
                )
    for numeric_column in ["Age", "ROSC"]:
        if numeric_column in modeling_dataframe.columns:
            indicator_column = f"{numeric_column}_missing"
            modeling_dataframe[indicator_column] = (
                modeling_dataframe[numeric_column].isna().astype(int)
            )
            fill_value = numeric_fill_values[numeric_column]
            modeling_dataframe[numeric_column] = modeling_dataframe[
                numeric_column
            ].fillna(fill_value)
    return modeling_dataframe, numeric_fill_values

In [21]:
repo_root = Path.cwd()
if repo_root.name == "models":
    repo_root = repo_root.parent
training_root_path = repo_root / "icare_data" / "training"
training_dataframe = build_training_dataframe(training_root_path)

In [22]:
csv = repo_root / "analysis" / "EfficientNet-B0" / "test_predictions_EfficientNetB0_grid(1).csv"
cnn_df_b0 = pd.read_csv(csv)

csv = repo_root / "analysis" / "EfficientNetV2-S_data" / "test_predictions_EfficientNetV2-S_grid(3).csv"
cnn_df_v2_s = pd.read_csv(csv)

In [23]:
# Aggregate CNN outputs per patient
cnn_patient_df_b0 = cnn_df_b0.groupby("Patient").agg({
    "prob_poor": "mean",
    "prob_good": "mean",
    "predicted_label": "mean"
}).reset_index()

cnn_patient_df_v2_s = cnn_df_v2_s.groupby("Patient").agg({
    "prob_poor": "mean",
    "prob_good": "mean",
    "predicted_label": "mean"
}).reset_index()

print(cnn_patient_df_v2_s.head())
print("Shape:", cnn_patient_df_v2_s.shape)

   Patient  prob_poor  prob_good  predicted_label
0      296   0.708546   0.291454         1.000000
1      299   0.212166   0.787834         0.285714
2      332   0.634508   0.365492         0.714286
3      342   0.919661   0.080339         1.000000
4      346   0.914135   0.085865         1.000000
Shape: (120, 4)


In [24]:
# Rename CNN output columns to avoid conflicts, then merge on Patient
cnn_patient_df_b0 = cnn_patient_df_b0.rename(columns={
    "prob_poor": "prob_poor_b0",
    "prob_good": "prob_good_b0",
    "predicted_label": "predicted_label_b0",
})

cnn_patient_df_v2_s = cnn_patient_df_v2_s.rename(columns={
    "prob_poor": "prob_poor_v2_s",
    "prob_good": "prob_good_v2_s",
    "predicted_label": "predicted_label_v2_s",
})

cnn_patient_merged = pd.merge(
    cnn_patient_df_b0,
    cnn_patient_df_v2_s,
    on="Patient",
    how="inner",
)

In [25]:
training_dataframe["Patient"] = training_dataframe["source_file"].str.extract(r'(\d+)').astype(int)

merged_df = training_dataframe.merge(
    cnn_patient_merged,
    on="Patient",
    how="inner"
)

print(merged_df.head())

   Patient Hospital   Age     Sex  ROSC   OHCA Shockable Rhythm   TTM Outcome  \
0      296        A  48.0    Male   NaN   True             True  36.0    Good   
1      299        A  45.0    Male   NaN   True             True  33.0    Good   
2      332        D  68.0  Female  60.0  False            False  33.0    Good   
3      342        B  71.0    Male  20.0  False            False  33.0    Poor   
4      346        E  55.0    Male  15.0   True            False  33.0    Poor   

   CPC                                        source_file  prob_poor_b0  \
0    1  c:\Users\zabit\Documents\GitHub\predicting-neu...      0.444388   
1    1  c:\Users\zabit\Documents\GitHub\predicting-neu...      0.329939   
2    1  c:\Users\zabit\Documents\GitHub\predicting-neu...      0.718341   
3    5  c:\Users\zabit\Documents\GitHub\predicting-neu...      0.923097   
4    5  c:\Users\zabit\Documents\GitHub\predicting-neu...      0.909905   

   prob_good_b0  predicted_label_b0  prob_poor_v2_s  prob_good

In [26]:
# STEP 5: Prepare data
model_df = merged_df.copy()

# Select features
features = [
    "Age", "Sex", "ROSC", "OHCA", "Shockable Rhythm", "TTM",
    "prob_poor_b0", "prob_good_b0", "predicted_label_b0", "prob_poor_v2_s", "prob_good_v2_s", "predicted_label_v2_s"
]

# Drop missing target
model_df = model_df.dropna(subset=["Outcome"])

# Encode categorical variables
categorical_cols = ["Sex", "OHCA", "Shockable Rhythm", "TTM"]

for col in categorical_cols:
    model_df[col] = model_df[col].astype(str)
    model_df[col] = LabelEncoder().fit_transform(model_df[col])

# Fill numeric missing
model_df["Age"] = model_df["Age"].fillna(model_df["Age"].median())
model_df["ROSC"] = model_df["ROSC"].fillna(model_df["ROSC"].median())

# STEP 6: Train model
X = model_df[features]
y = model_df["Outcome"]

# Encode target
y = LabelEncoder().fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=1234, stratify=y
)

# Improved model
model = RandomForestClassifier(
    n_estimators=300,
    max_depth=10,
    min_samples_split=5,
    random_state=123
)

model.fit(X_train, y_train)

# Predictions
y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)

print("\n FINAL MODEL ACCURACY:", accuracy)


 FINAL MODEL ACCURACY: 0.6666666666666666


In [27]:
param_grid = {
    "n_estimators": [100, 300],
    "max_depth": [None, 10, 20],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [2, 4],
    "max_features": ["sqrt"],
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=1234)

model = RandomForestClassifier(random_state=42, n_jobs=-1)

grid_search = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    scoring="accuracy",
    cv=cv,
    n_jobs=-1,
    verbose=1,
)

grid_search.fit(X, y)

print("Best CV accuracy:", grid_search.best_score_)
print("Best hyperparameters:", grid_search.best_params_)

best_model = grid_search.best_estimator_

scoring = {
    "accuracy": "accuracy",
    "auc": "roc_auc",
    "sensitivity": "recall",
    "specificity": make_scorer(recall_score, pos_label=0),
    "precision": "precision",
    "f1": "f1",
}

cv_results = cross_validate(
    best_model,
    X,
    y,
    cv=cv,
    scoring=scoring,
    n_jobs=-1,
)

summary_rows = []
for metric in ["accuracy", "auc", "sensitivity", "specificity", "precision", "f1"]:
    values = cv_results[f"test_{metric}"]
    summary_rows.append(
        {
            "Metric": metric.capitalize() if metric != "auc" else "AUC",
            "Mean ± SD": f"{values.mean():.3f} ± {values.std():.3f}",
            "Min": f"{values.min():.3f}",
            "Max": f"{values.max():.3f}",
        }
    )

summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_string(index=False))


Fitting 5 folds for each of 36 candidates, totalling 180 fits
Best CV accuracy: 0.7416666666666667
Best hyperparameters: {'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 2, 'min_samples_split': 2, 'n_estimators': 100}
     Metric     Mean ± SD   Min   Max
   Accuracy 0.742 ± 0.031 0.708 0.792
        AUC 0.818 ± 0.043 0.733 0.852
Sensitivity 0.787 ± 0.088 0.667 0.933
Specificity 0.667 ± 0.070 0.556 0.778
  Precision 0.799 ± 0.019 0.778 0.833
         F1 0.790 ± 0.037 0.741 0.848


In [28]:
best_model

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",2
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric

In [29]:
X

,Age,Sex,ROSC,OHCA,Shockable Rhythm,TTM,prob_poor_b0,prob_good_b0,predicted_label_b0,prob_poor_v2_s,prob_good_v2_s,predicted_label_v2_s
0,48.0,1,19.0,1,1,1,0.444388,0.555612,0.500000,0.708546,0.291454,1.000000
1,45.0,1,19.0,1,1,0,0.329939,0.670061,0.333333,0.212166,0.787834,0.285714
2,68.0,0,60.0,0,0,0,0.718341,0.281659,1.000000,0.634508,0.365492,0.714286
3,71.0,1,20.0,0,0,0,0.923097,0.076903,1.000000,0.919661,0.080339,1.000000
4,55.0,1,15.0,1,0,0,0.909905,0.090095,1.000000,0.914135,0.085865,1.000000
...,...,...,...,...,...,...,...,...,...,...,...,...
115,57.0,0,19.0,1,1,0,0.302176,0.697824,0.500000,0.471792,0.528208,0.500000
116,82.0,0,19.0,1,1,0,0.127430,0.872570,0.000000,0.264751,0.735249,0.000000
117,54.0,1,38.0,1,0,0,0.702642,0.297358,1.000000,0.719650,0.280350,1.000000
118,54.0,1,21.0,1,0,0,0.736167,0.263833,1.000000,0.811099,0.188901,1.000000


In [30]:
y

array([0, 0, 0, 1, 1, 0, 1, 0, 1, 0, 1, 1, 1, 0, 1, 0, 1, 0, 0, 1, 1, 1,
       1, 0, 1, 0, 1, 0, 1, 0, 1, 1, 0, 1, 0, 0, 1, 1, 1, 1, 1, 0, 1, 1,
       1, 1, 1, 1, 1, 1, 0, 1, 0, 1, 1, 0, 0, 0, 0, 1, 1, 1, 1, 0, 0, 0,
       1, 1, 1, 1, 1, 0, 1, 0, 1, 0, 1, 1, 1, 1, 0, 0, 1, 1, 0, 1, 0, 1,
       1, 1, 0, 1, 1, 1, 0, 0, 0, 1, 0, 1, 1, 0, 0, 1, 1, 1, 1, 1, 0, 0,
       1, 0, 1, 1, 1, 0, 1, 1, 1, 1])

In [31]:
merged_df.tail(6)

,Patient,Hospital,Age,Sex,ROSC,OHCA,Shockable Rhythm,TTM,Outcome,CPC,source_file,prob_poor_b0,prob_good_b0,predicted_label_b0,prob_poor_v2_s,prob_good_v2_s,predicted_label_v2_s
114,953,A,74.0,Male,NaN,True,True,33.0,Poor,3,c:\Users\zabit\Documents\GitHub\predicting-neu...,0.468893,0.531107,0.5,0.574069,0.425931,0.5
115,963,A,57.0,Female,NaN,True,True,33.0,Good,1,c:\Users\zabit\Documents\GitHub\predicting-neu...,0.302176,0.697824,0.5,0.471792,0.528208,0.5
116,975,A,82.0,Female,NaN,True,True,33.0,Poor,5,c:\Users\zabit\Documents\GitHub\predicting-neu...,0.127430,0.872570,0.0,0.264751,0.735249,0.0
117,976,B,54.0,Male,38.0,True,False,33.0,Poor,5,c:\Users\zabit\Documents\GitHub\predicting-neu...,0.702642,0.297358,1.0,0.719650,0.280350,1.0
118,981,D,54.0,Male,21.0,True,False,33.0,Poor,5,c:\Users\zabit\Documents\GitHub\predicting-neu...,0.736167,0.263833,1.0,0.811099,0.188901,1.0
119,997,D,26.0,Female,NaN,True,False,NaN,Poor,5,c:\Users\zabit\Documents\GitHub\predicting-neu...,0.682245,0.317755,1.0,0.781715,0.218285,1.0


In [32]:
# Work on a copy for preprocessing, keep merged_df for final assignment
df_pred = merged_df.copy()

# Check all required features exist
missing = [c for c in features if c not in df_pred.columns]
if missing:
    raise KeyError(f"Missing columns required by model: {missing}")

# Categorical encoding (same approach as training)
categorical_cols = ["Sex", "OHCA", "Shockable Rhythm", "TTM"]
encoders = {}
for col in categorical_cols:
    if col in df_pred.columns:
        df_pred[col] = df_pred[col].astype(str)
        le = LabelEncoder()
        df_pred[col] = le.fit_transform(df_pred[col])
        encoders[col] = le

# Numeric fills (use medians from the current dataframe; replace with training medians if available)
for col in ["Age", "ROSC"]:
    if col in df_pred.columns:
        df_pred[col] = df_pred[col].fillna(df_pred[col].median())

# Prepare feature matrix and predict
X_pred = df_pred[features]
y_pred = best_model.predict(X_pred)
merged_df["rf_pred"] = y_pred  # integer class as used by the model

# Add class probabilities if model supports it
if hasattr(best_model, "predict_proba"):
    proba = best_model.predict_proba(X_pred)
    classes = best_model.classes_
    for i, cls in enumerate(classes):
        merged_df[f"rf_proba_{cls}"] = proba[:, i]

# If you saved the target LabelEncoder during training as `target_le`, restore original labels:
if "target_le" in globals():
    merged_df["rf_pred_label"] = globals()["target_le"].inverse_transform(merged_df["rf_pred"])

print("Added columns: rf_pred", [f"rf_proba_{c}" for c in getattr(best_model, "classes_", [])])

Added columns: rf_pred ['rf_proba_0', 'rf_proba_1']


In [35]:
output_path = repo_root / "analysis" / "RF" / "random_forest_predictions.csv"
output_path.parent.mkdir(parents=True, exist_ok=True)
merged_df.to_csv(output_path, index=False)
merged_df.head()

,Patient,Hospital,Age,Sex,ROSC,OHCA,Shockable Rhythm,TTM,Outcome,CPC,source_file,prob_poor_b0,prob_good_b0,predicted_label_b0,prob_poor_v2_s,prob_good_v2_s,predicted_label_v2_s,rf_pred,rf_proba_0,rf_proba_1
0,296,A,48.0,Male,NaN,True,True,36.0,Good,1,c:\Users\zabit\Documents\GitHub\predicting-neu...,0.444388,0.555612,0.500000,0.708546,0.291454,1.000000,0,0.738500,0.261500
1,299,A,45.0,Male,NaN,True,True,33.0,Good,1,c:\Users\zabit\Documents\GitHub\predicting-neu...,0.329939,0.670061,0.333333,0.212166,0.787834,0.285714,0,0.928726,0.071274
2,332,D,68.0,Female,60.0,False,False,33.0,Good,1,c:\Users\zabit\Documents\GitHub\predicting-neu...,0.718341,0.281659,1.000000,0.634508,0.365492,0.714286,0,0.606202,0.393798
3,342,B,71.0,Male,20.0,False,False,33.0,Poor,5,c:\Users\zabit\Documents\GitHub\predicting-neu...,0.923097,0.076903,1.000000,0.919661,0.080339,1.000000,1,0.007333,0.992667
4,346,E,55.0,Male,15.0,True,False,33.0,Poor,5,c:\Users\zabit\Documents\GitHub\predicting-neu...,0.909905,0.090095,1.000000,0.914135,0.085865,1.000000,1,0.000000,1.000000
